In [1]:
# Import necessary libraries
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
from pynput import keyboard
import threading



# Uncomment one of the following lines depending on your setup

# If you are using the real car, uncomment the next lines and comment the simulator lines
from serial import Serial
# import sounddevice as sd

# If you are using the simulator, uncomment the next lines and comment the real car lines
# from KITT_Simulator.serial_simulator import Serial
from KITT_Simulator.sounddevice_simulator import sounddevice as sd

# Note: After changing the import statement, you need to restart the kernel for changes to take effect.

Run below to get all bluetooth ports. 
To connect: turn on bluetooth, Windows settings > Bluetooth & devices > Devices > More bluetooth settings > COM ports


Run cell with `serial = Serial('COM#', 115200)` to connect with KITT.

In [2]:
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
for port in ports:
    print(port.device)

COM7
COM3
COM6
COM5


In [3]:
class KITT:
    def __init__(self, port, baudrate=115200):
        self.serial = Serial(port, baudrate, rtscts=True, write_timeout=1)
        
        # Initialize beacon parameters here using send_command
        self.send_command("S")

        time.sleep(1)  # Wait for the serial connection to stabilize

        
        # Check if we are on Real Hardware (which has .in_waiting)
        if hasattr(self.serial, 'in_waiting'):
            if self.serial.in_waiting > 0:
                self.serial.read(self.serial.in_waiting)
                
        # Otherwise, we are in the Simulator
        else:
            # The simulator stores the 'S' response as a single list item.
            # We call read_until once to 'pop' this item and discard it.
            try:
                self.serial.read_until(b'\n') 
            except Exception:
                pass # Ignore if buffer was already empty




        # self.serial.read_until(b'\n')  # Clear initial



        # Set carrier frequency, bit frequency, repetition count, and code pattern
        self.send_command(f'F10000\n')
        self.send_command(f'B5000\n')
        self.send_command(f'R2500\n')
        self.send_command(f'C00000000\n')

        self.distance_list = []

        pass  


    def send_command(self, command):
        self.serial.write(command.encode())
        pass

    def set_speed(self, speed):
        self.send_command(f'M{speed}\n')
        pass

    def set_angle(self, angle):
        self.send_command(f'D{angle}\n')
        pass

    def stop(self):
        self.set_speed(150)
        self.set_angle(150)
        pass

    def start_beacon(self):
        # Send commands to start the beacon
        # Use the command 'A1\n'
        self.send_command('A1\n')
        pass

    def stop_beacon(self):
        # Send commands to stop the beacon
        # Use the command 'A0\n'
        self.send_command('A0\n')
        pass

    

    ### Add method read_distance_sensors() in here
    def read_distance_sensors(self):
        
        
        # self.send_command(f"Sd\n")
        ts = time.time()

        

        try:
            self.serial.write(b"Sd\n")
            
            status = self.serial.read_until(b'\x04').decode('utf-8').strip()
            # if not status:
            #     return ts, None, None
            # print(status)

        # Initialize variables to hold distance values
            dist_L = None
            dist_R = None
        
            # print(status)
            # lines = status.splitlines()
            # print(array)


            # ########################################
            # ## using serial.write(b"S\n") for data request:
            # # Iterate over each line to find distance data
            # for line in lines:
            #     if "Dist." in line:
            #         # Split the line into words
            #         words = line.split()
            #         # Extract distance values based on their positions
            #         L = float(words[3]) # Number corresponding to left sensor is located at index 3
            #         R = float(words[5]) # Number corresponding to right sensor is located at index 5


            #         # Assign dist_L and dist_R accordingly
            #         dist_L = round(L, 3)
            #         dist_R = round(R, 3)
            #         break  # Exit the loop after finding the distances
            #     else:
            #         pass
            # ########################################



            ####################################### 
            ### using serial.write(b"Sd\n") for data request:
            # Initialize variables to hold distance values
            
            # print(status)
            lines = status.splitlines()
            # print(lines)
            for word in lines:
                if "L" in word:
                    try:
                        value = float(word[3:])
                        dist_L = round(value, 3)
                    except ValueError:
                        pass
                elif "R" in word:
                    try:
                        value = float(word[3:])
                        dist_R = round(value, 3)
                    except ValueError:
                        pass
                else:
                    pass
            #######################################
        


            # print(dist_L, dist_R)
            # print(status)
            self.distance_list.append([ts,dist_L, dist_R])
            return ts, dist_L, dist_R

        except Exception as e:
            print("bleh")
            # # Handle any exceptions
            # print(f"An error occurred: {e}")
            # return ts, None, None
    
        
    def close(self):
        # Close the serial connection
        self.serial.close()
        pass

    def __del__(self):
        self.serial.close()


In [4]:
serial = Serial('COM7', 115200)

# serial = Serial('COM7', 115200)  #Uncomment this when using real KITT


serial.write(b"M155\n")
serial.write(b"D150\n")
time.sleep(3)


serial.close()

In [5]:
### Student Version ###

def wasd_control(kitt):
    print("Control the car with W (forward), S (backward), A (left), D (right), E (start beacon), R (stop beacon), and Q (to exit).")

    car_speed = 150  # Neutral speed
    car_steering = 150  # Neutral steering
    neutral_speed = 150  # Target speed when no throttle input is held
    neutral_steering = 150 # Target steering when no direction input is held
    speed_step = 2
    decay_step = 2*5

    

    def on_press(key):
        try:
            pressed.add(key.char)
        except:
            pass
    def on_release(key):
        try:
            pressed.discard(key.char)
        except:
            pass
        if key.char =='q':
            return False
    pressed = set()
    listener = keyboard.Listener(on_press=on_press, on_release = on_release)
    listener.start()
    try:
        while listener.is_alive():
            # Apply throttle adjustments
            if 'w' in pressed:
                car_speed += speed_step
            elif 's' in pressed:
                car_speed -= speed_step
            else:
                # Drop speed back toward neutral when no throttle input is held
                if car_speed > neutral_speed:
                    car_speed = max(car_speed - decay_step, neutral_speed)
                elif car_speed < neutral_speed:
                    car_speed = min(car_speed + decay_step, neutral_speed)
                if car_steering > neutral_steering:
                    car_steering = max(car_steering - decay_step, neutral_steering)
                elif car_steering < neutral_steering:
                    car_steering = min(car_steering + decay_step, neutral_steering)

            # Steering adjustments
            if 'a' in pressed:
                car_steering += 10
            elif 'd' in pressed:
                car_steering -= 10

            # Beacon controls
            if 'e' in pressed:
                kitt.start_beacon()
            elif 'r' in pressed:
                kitt.stop_beacon()

            # Ensure speed and steering values are within valid ranges
            car_speed = min(max(car_speed, 135), 165)
            car_steering = min(max(car_steering, 100), 200)

            # Send the speed and angle to KITT
            kitt.set_speed(car_speed)
            kitt.set_angle(car_steering)
            time.sleep(0.08)
    except Exception as e:
        # Handle any exceptioqns
        print(f"An error occurred: {e}")
    finally:
        # Ensure the car is stopped and serial connection is closed
        kitt.stop()
        kitt.close()

In [ ]:
# num = 343.23948324
# # nom = "{:10.4f}".format(num)
# nom = round(num, 3)
# print(nom)

# idiot = '**************************\n'
# idiot += '* Sensors:\n'
# idiot += '* Dist. L {} R {}\n'.format(20.324,76.435)
# idiot += '* V_batt {} V\n'.format(2)
# # line = '* Dist. L {} R {}\n'.format(20.324,76.435)
# idiot1 = idiot.split()

# for shit in idiot1:

#     print(shit)
#     if shit.startswith("L"):
#         val = float(shit[1:])
#         dist_L = round(val,3)

#     elif shit.startswith("R"):
#         # val = float(shit[1:])
#         dist_R = round(val,3)

# print(dist_L, dist_R)
# print("It's working")

In [ ]:
### Student Version ###


if __name__ == "__main__":
    # Create an instance of KITT with the correct serial port
    # Replace '/dev/ttyUSB0' with your actual serial port
    kitt = KITT('COM7', 115200)
    time.sleep(1)


    ##### uncomment this if using wasd_control
    tthread = threading.Thread(target=wasd_control, args=(kitt,))
    tthread.daemon=True
    tthread.start()

    motor_speed_value = 160
    # write_distance_sensors(kitt)
    # Initialize a list to store recorded data
    data = []
    # Record data for a specified duration (e.g., 10 seconds)
    # recording_duration = 10  # in seconds
    start_time = time.time()


    try:
        while tthread.is_alive():  #### uncomment this if using wasd_control
        # while time.time() - start_time< recording_duration: #### uncomment this if NOT using wasd_control
            ts, dist_L, dist_R = kitt.read_distance_sensors()
            print(dist_L, dist_R, "waddup")
            # Record current time and distances
            current_time = ts-start_time
            # data.append([current_time, dist_L, dist_R])

            if dist_L is not None and dist_R is not None:
                # data.append([current_time, dist_L, dist_R])
                
                print("ur gay")

                # if dist_L < 40.0 or dist_R < 40.0: 

                #     kitt.set_speed(135)
                #     print(f"Stopping: L={dist_L}, R={dist_R}")
                #     print(f"stopping distance reached at t = {current_time}")
                #     time.sleep(0.4)
                #     kitt.set_speed(150)

                # Remove the last entry of the data before running the next loop, stopping data (Remove duplicate)
                
                data.pop(len(data)-1)

                # break
            #     pass
            else: pass

            time.sleep(0.1)

        tthread.join() #### uncomment this if using wasd_control

        # Note: you can also add a small loop here and still read the stopping data
        while tthread.is_alive(): #### uncomment this if using wasd_control
        # while time.time() - start_time< recording_duration: #### uncomment this if NOT using wasd_control
            ts, dist_L, dist_R = kitt.read_distance_sensors()
            
                
            current_time = round(ts-start_time,3)
            data.append([current_time, dist_L, dist_R])
            print(data)

            # stop if car hits the wall or goes out of bounds. This will show only the real data in the csv file (no 'inf' data)
            if dist_L == float('inf') or dist_R == float('inf'):
                data.pop(len(data)-1)
                break
            else: pass

            time.sleep(0.1)  # Wait before the next reading

        tthread.join() #### uncomment this if using wasd_control

    finally:
        print(data, dist_R, dist_L, current_time)
        # kitt.stop()
        kitt.close()


    
        # Remove the first few readings from the data as they might be inaccurate
        data = data[3:]

        # Write the recorded data to a CSV file
        # Recommeded file output: Files/Recordings/kitt_distance_data_{speed}.csv

        with open(f"files/Recordings/kitt_distance_data_{motor_speed_value}.csv", "w", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["Time", "Distance_L", "Distance_R"])
            writer.writerows(data)


        
#wwwwwssswwwssssssqwswwwsqwsssssssswwwwwwwwwwwwwwwwwsqsssswwwwwws

Control the car with W (forward), S (backward), A (left), D (right), E (start beacon), R (stop beacon), and Q (to exit).
0.0 0.0 waddup
ur gay
[] 0.0 0.0 0.0


IndexError: pop from empty list

An error occurred: Attempting to use a port that is not open


Exception in thread Thread-5 (wasd_control):
Traceback (most recent call last):
  File "C:\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\pavan\AppData\Local\Temp\ipykernel_12064\950723749.py", line 73, in wasd_control


  File "C:\Users\pavan\AppData\Local\Temp\ipykernel_12064\2896935551.py", line 56, in stop
  File "C:\Users\pavan\AppData\Local\Temp\ipykernel_12064\2896935551.py", line 48, in set_speed
  File "C:\Users\pavan\AppData\Local\Temp\ipykernel_12064\2896935551.py", line 44, in send_command
  File "c:\VSCode\IP3\.venv\Lib\site-packages\serial\serialwin32.py", line 306, in write
    raise PortNotOpenError()
serial.serialutil.PortNotOpenError: Attempting to use a port that is not open
Unhandled exception in listener callback
Traceback (most recent call last):
  File "c:\VSCode\IP3\.venv\Lib\site-packages\pynput\_util\__init__.py", line 230, in inner
    return f(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\VSCode\IP3\.venv\Lib\site-packages\pynput\keyboard\_win32.py", line 332, in _process
    self.on_release(key, injected)
  File "c:\VSCode\IP3\.venv\Lib\site-packages\pynput\_util\__init__.py", line 146, in inner
    if f(*args) is False:
       ^^^^^^^^
  File "c:\VS

In [ ]:
serial = Serial('COM7', 115200)

# serial = Serial('COM7', 115200)  #Uncomment this when using real KITT

# time.sleep(1)
serial.write(b"M140\n")
serial.write(b"D150\n")
time.sleep(2.5)


serial.close()

In [ ]:
kitt = KITT('/dev/ttyUSB0')

# serial = serial()

# Send test commands
# tthread = threading.Thread(target=wasd_control, args=(kitt,))
# tthread.start()


# write_distance_sensors(kitt)


# time.sleep(1)


# Initialize a list to store recorded data
data = []
# Record data for a specified duration (e.g., 10 seconds)
recording_duration = 10  # in seconds
start_time = time.time()
motor_speed_value = 160
kitt.set_speed(160)
kitt.set_angle(150)
time.sleep(3)

kitt.close()

In [ ]:
def print_distance_sensors(motor_speed_value):


    csv_filename = f'Files/Recordings/kitt_distance_data_{motor_speed_value}.csv'
    data = pd.read_csv(csv_filename)

    # TODO: Remove duplicate time stamps and merge L and R data
    # Create a new DataFrame to hold your processed data
    nmerged_data = []
    merged_data = []


    # Iterate over the data
    for index, row in data.iterrows():
        
        # Extract time and distances
        time_stamp = row['Time']
        dist_L = row['Distance_L']
        dist_R = row['Distance_R']

        # print(index)
        
        # TODO: Decide which distance to use or how to merge them

        distance = min(dist_L, dist_R)

        nmerged_data.append([time_stamp, dist_L, dist_R])
        merged_data.append([time_stamp, distance])
        # print(time_stamp)
        # print(distance, "\n")



    # Convert merged data to DataFrame
    nmerged_df = pd.DataFrame(nmerged_data, columns=['Time', 'Distance_L', "Distance_R"])
    merged_df = pd.DataFrame(merged_data, columns=['Time', "Distance"])

    # keep only the first row where the 'Distance' value changes.
    filtered_df = merged_df.drop_duplicates(subset=['Distance'], keep='first').copy()






    # ... (Your existing code up to creating filtered_df) ...

    # 1. Calculate Instantaneous Velocity (The "Noisy" one you have now)
    # We use .shift() to access the previous row safely
    dt = filtered_df['Time'].diff()
    dd = -filtered_df['Distance'].diff() # Negative because distance decreases as we move forward
    
    filtered_df['Velocity_Time'] = filtered_df['Time'] - (dt / 2)
    filtered_df['Velocity_Time'] = filtered_df['Velocity_Time'].dropna()


    filtered_df['Velocity_Raw'] = dd / dt

    # 2. THE FIX: Apply a Rolling Average Filter
    # This looks at the last 'N' measurements and averages them. 
    # A window of 5-10 is usually a good balance between smoothness and responsiveness.
    window_size = 10 
    filtered_df['Velocity_Smoothed'] = filtered_df['Velocity_Raw'].rolling(window=window_size, center=True).mean()

    # Clean up NaNs created by diff and rolling
    plot_data = filtered_df.dropna()

    # make sure plot starts from first time interval in the csv file, even if velocity is zero there
    plot_data = plot_data[plot_data['Velocity_Time'] >= merged_df['Time'].min()]



    # # TODO: calculate velocity (change in distance over change in time)
    # filtered_df['Velocity'] = -filtered_df['Distance'].diff() / filtered_df['Time'].diff()
    # filtered_df['Velocity'] = filtered_df['Velocity'].dropna()


    # # Calculate the time corresponding to each velocity estimate
    # # It's common to use the midpoint of the time intervals

    # filtered_df['Velocity_Time'] = (filtered_df['Time'] - filtered_df['Time'].diff() / 2)
    # filtered_df['Velocity_Time'] = filtered_df['Velocity_Time'].dropna()


    # print(time_stamp)

    # Plotting Distance for each sensor seperately (L and R)
    plt.figure()
    plt.plot(merged_df['Time'], nmerged_df['Distance_L'], label='Distance to wall (left sensor)')
    plt.plot(merged_df["Time"], nmerged_df['Distance_R'], label='Distance to wall (right sensor)')
    plt.xlabel('Time (s)')
    plt.ylabel('Distance (cm)')
    plt.title('Distance to Wall Over Time')
    plt.grid(True)
    plt.legend()
    plt.show()


    # Plotting Distance merged
    plt.figure()
    plt.plot(merged_df['Time'], merged_df['Distance'], label='Distance to wall (merged)')
    plt.xlabel('Time (s)')
    plt.ylabel('Distance (cm)')
    plt.title('Distance to Wall Over Time')
    plt.grid(True)

    # plt.xlim(0.8,1.2)
    plt.xlim(0,time_stamp)

    plt.legend()
    plt.show()




    # Plotting Velocity
    plt.figure(figsize=(10, 6))
    
    # Plot the raw noisy data lightly in the background to compare
    plt.plot(plot_data['Velocity_Time'], plot_data['Velocity_Raw'], 
             label='Raw Instantaneous', color='red', alpha=0.2)
             
    # Plot the new SMOOTHED velocity
    plt.plot(plot_data['Velocity_Time'], plot_data['Velocity_Smoothed'], 
             label=f'Smoothed (Window={window_size})', color='blue', linewidth=2)

    plt.xlabel('Time (s)')
    plt.ylabel('Velocity (cm/s)')
    plt.title('Velocity of KITT Over Time')
    plt.grid(True)
    plt.xlim(0, time_stamp)
    plt.legend()
    plt.show()




    # # Plotting Velocity
    # plt.figure()

    # # plt.plot(merged_df['Velocity_Time'], merged_df['Velocity'], label='Velocity (cm/s)')
    # plt.plot(filtered_df['Velocity_Time'], filtered_df['Velocity'], label='Velocity (cm/s)')

    # plt.xlabel('Time (s)')
    # plt.ylabel('Velocity (cm/s)')
    # plt.title('Velocity of KITT Over Time')
    # plt.grid(True)

    # # plt.xlim(1.5,2)
    # plt.xlim(0,time_stamp)

    # plt.legend()
    # plt.show()


print_distance_sensors(motor_speed_value)

In [ ]:
for i, device in enumerate(sd.query_devices()):
   print(i, device['name'])


In [ ]:
Fs = 48000 # Sample rate
duration = 5 # duration of recording in seconds
N = int(Fs * duration) # amount of samples


In [ ]:
sd.default.device = 15   # device ID
sd.default.samplerate = Fs

samples = sd.rec(N, samplerate=Fs, channels=5) # start recording
sd.wait() # wait untill all samples are captured

print(samples.shape) # will be N x 5

In [ ]:
### Student Version ###

print(samples[:50],"\n")

# TODO: Reshape the data into a matrix with 5 columns (one for each microphone)
def mic_matrix(data) -> np.ndarray:

    array = np.array(samples)

    matrix = array.reshape((-1, 5))

    return matrix


audio_data = mic_matrix(samples)

# print(audio_data[:50])
print(audio_data.shape)
# print(len(audio_data))
# print(audio_data.shape[0])

In [ ]:
### Student Version ###

# TODO: Plot the data for each microphone

fig, axes = plt.subplots(nrows=5,figsize=(20,18))
time = np.arange(len(audio_data))/Fs
# time = np.linspace(0, len(audio_data), Fs)

for i in range(5):
    ax = axes[i]
    ax.plot(time, audio_data[:,i])

plt.show()

